In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("incremental_run","0")


In [0]:
incremental_load = dbutils.widgets.get("incremental_run")
print(incremental_load)

In [0]:
df = spark.sql('''
               SELECT * FROM parquet.`abfss://silver@storageforcarsproject.dfs.core.windows.net/OBT/`''')

display(df)

In [0]:
df_src = spark.sql('''
                   SELECT DISTINCT(Model_ID),model_subcat FROM PARQUET.`abfss://silver@storageforcarsproject.dfs.core.windows.net/OBT/`''')
display(df_src)

In [0]:
if spark.catalog.tableExists("cars_cata.gold.dim_model"):
    df_sink = spark.sql('''
                        SELECT dim_model_key,Model_ID,model_subcat FROM cars_cata.gold.dim_model''')
else:
    df_sink = spark.sql('''
                        SELECT 1 as dim_model_key,Model_id,model_subcat FROM parquet.`abfss://silver@storageforcarsproject.dfs.core.windows.net/OBT/` WHERE 1=0''')

In [0]:
df_total = df_src.join(df_sink,df_src["Model_ID"]==df_sink["Model_ID"],"LEFT").select(df_src["Model_ID"],df_src["model_subcat"],df_sink["dim_model_key"])

### # New Record

In [0]:
df_new = df_total.filter(col("dim_model_key").isNull()).select(df_src["Model_ID"],df_src["model_subcat"])

### Old Record

In [0]:
df_old = df_total.filter(col("dim_model_key").isNotNull())

### ## surrogate key for new record

In [0]:
if incremental_load == '0':
    max_value = 1
else:
    max_value_df = spark.sql('''
                             SELECT max(dim_model_key) FROM cars_cata.gold.dim_model''')
    max_value = max_value_df.collect()[0][0] + 1


In [0]:
df_new = df_new.withColumn('dim_model_key',max_value + monotonically_increasing_id())


In [0]:
df_final = df_new.union(df_old)


## SCD TYPE1

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("cars_cata.gold.dim_model"):
    delta_obj = DeltaTable.forPath(spark,"abfss://gold@storageforcarsproject.dfs.core.windows.net/dim_model")
    delta_obj.alias("trg").merge(df_final.alias("src"),"trg.Model_ID == src.Model_ID")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
else:
    df_final.write.format("delta")\
        .mode("overwrite")\
        .option("path","abfss://gold@storageforcarsproject.dfs.core.windows.net/dim_model")\
        .saveAsTable("cars_cata.gold.dim_model")
        

In [0]:
%sql
select * FROM delta.`abfss://gold@storageforcarsproject.dfs.core.windows.net/dim_model`